# CVRP Algorithm Selection

Compact fast-mode notebook for the frozen final experiment. It loads saved artifacts instead of rerunning the expensive solver experiment.

## 1. Scope

Symmetric Euclidean CVRP, one depot, homogeneous capacity, positive heterogeneous demands, routes returning to the depot.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('reports/final_ml_experiment')
assert BASE.exists(), BASE
summary = json.loads((BASE / 'ml_summary.json').read_text())
metrics = pd.read_csv(BASE / 'model_metrics.csv')
experiment = pd.read_csv(BASE / 'experiment_table.csv')
features = json.loads((BASE / 'feature_columns.json').read_text())
summary['n_rows'], summary['train_rows'], summary['test_rows']


## 2. Final Portfolio

The main selector uses five VeRyPy-based heuristics. Gaskell remains implemented/audited but is excluded from ML because it won too few development instances.

In [ ]:
summary['label_counts']


## 3. Distance Convention

The final experiment uses `cvrplib_euc_2d`, the rounded CVRPLIB/TSPLIB convention. Costs and labels are not mixed with floating Euclidean values.

In [ ]:
experiment['distance_convention'].value_counts()


## 4. Dataset Design

The final synthetic dataset is a factorial-style grid over spatial family, depot position, demand regime, customer count, and seed.

In [ ]:
experiment[['spatial_family','depot_position','demand_regime','customer_count','seed']].nunique()


## 5. Feature Matrix

Model features are pre-solution instance features only. IDs, seeds, generator labels, solver costs, runtimes, feasibility columns, and labels are excluded.

In [ ]:
len(features), features[:10]


## 6. Leakage Check

This fast check confirms prohibited solver/outcome columns are not in the saved feature manifest.

In [ ]:
prohibited_fragments = ['cost', 'runtime', 'feasible', 'failure', 'best_solver', 'best_cost', 'instance_id', 'seed', 'spatial_family', 'depot_position', 'demand_regime']
bad = [c for c in features if any(p in c for p in prohibited_fragments)]
assert bad == [], bad
'no prohibited feature columns'


## 7. Label Distribution

![Label distribution](reports/final_ml_experiment/plots/label_distribution.png)

In [ ]:
pd.Series(summary['label_counts']).sort_values(ascending=False)


## 8. Baselines And Models

Evaluation includes Dummy classifiers, SBS selected from training data only, VBS oracle, KNN, Decision Tree, Random Forest, and Gradient Boosting.

In [ ]:
metrics[['method','accuracy','balanced_accuracy','macro_f1','mean_regret','mean_cost_ratio','within_1_percent','within_5_percent']]


## 9. Accuracy

![Accuracy and balanced accuracy](reports/final_ml_experiment/plots/accuracy_balanced_accuracy.png)

## 10. Routing Regret

![Mean regret](reports/final_ml_experiment/plots/mean_regret.png)

In [ ]:
metrics[['method','mean_regret','median_regret','p90_regret','p95_regret','max_regret']].sort_values('mean_regret')


## 11. Cost Ratio

![Mean cost ratio](reports/final_ml_experiment/plots/mean_cost_ratio.png)

## 12. Within-Threshold Quality

![Within thresholds](reports/final_ml_experiment/plots/within_thresholds.png)

## 13. Best Model

Random Forest is the best verified ML model by mean regret. It improves mean regret versus SBS while using only pre-solution instance features.

In [ ]:
metrics.loc[metrics['method'].isin(['RandomForest','SBS','VBS']), ['method','accuracy','mean_regret','within_1_percent','within_5_percent','mean_selected_solver_runtime']]


## 14. Confusion Matrix

![Random Forest confusion matrix](reports/final_ml_experiment/plots/best_model_confusion_matrix.png)

## 15. Feature Importance

![Random Forest feature importance](reports/final_ml_experiment/plots/random_forest_feature_importance.png)

In [ ]:
pd.read_csv(BASE / 'random_forest_feature_importance.csv').head(10)


## 16. Grouped Performance

Grouped metrics are saved for spatial family, depot position, demand regime, customer count, and true best solver.

In [ ]:
grouped = json.loads((BASE / 'grouped_metrics.json').read_text())
pd.DataFrame(grouped['customer_count'])


## 17. Solver Failures

Failures are recorded without aborting the experiment. The one Wren-Holliday timeout is handled by labelling from remaining feasible solvers.

In [ ]:
summary['solver_failure_counts']


## 18. Integrity Checks

These checks recompute the label-minimum rule and confirm VBS metrics.

In [ ]:
from cvrp_algorithm_selection.ml_experiment import final_solver_prefixes, validate_label_minima
validate_label_minima(experiment, final_solver_prefixes())
vbs = metrics[metrics['method'] == 'VBS'].iloc[0]
assert abs(vbs['mean_regret']) < 1e-12
assert abs(vbs['mean_cost_ratio'] - 1.0) < 1e-12
'integrity checks passed'


## 19. Fast Reproduction Command

Run tests with `uv run pytest`. Run this notebook in fast mode by executing its code cells against saved outputs; no solver regeneration is required.

## 20. Conclusion

The portfolio is complementary enough for supervised selection, and Random Forest beats SBS on mean routing regret. The strongest limitations are synthetic data, heuristic labels rather than exact optima, and runtime being secondary rather than part of the label objective.